# wf_v4.6 GPU session (diary Nov 16): loop 1 with LLM entity names, then loop 2 with Llama
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.6`, folder `step1a/`. Results in `results/wf_v46/`.

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON (setting `llama8b`). Runtime: **A100 (40 GB or more)**.
**One GPU session only**: the model loads once (about 4 min), then every job runs back to back. Expected model time under 10 min.

- Cells 1-8: setup. Cell 2 installs, then restart the session, rerun cell 1 and skip cell 2.
- Cell 4: GLiNER, deberta-v3-base and bge-m3 (loop-2 evidence check), downloaded before vLLM starts.
- Cell 5: loads Llama with `gpu_memory_utilization=0.6`, so bge-m3 and GLiNER fit next to it.
- Cell 9: **Y3** loop 1 with LLM entity names on raw dev50 and dev50b (Step 1a-2 from git; Step 3 v4.4, 3 orders; Step 4 v4.3 seed 0). About 450-600 calls per set.
- Cell 10: **Y5 + Y6** loop 2 on the Y3 output (Step 1b 3 orders + vote, evidence check, Step 3 naming). About 30-60 calls per set.
- Cell 11: **Y8** (only if time is left) Step 4 v4.3 with pass 2 in batches of 20, seed 0. About 10-20 calls per set.
- Cell 12: total GPU session time, then **Runtime > Disconnect and delete runtime**.
- Each run cell pushes its results to git. After a crash: rerun cells 1, 3-8 and the cell you were on (answered calls are cached).
- Stop rule: a cell stops by itself if a set needs more than MAX_CALLS calls; then tell Claude before going on.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.


In [ ]:
#@title 1. Tokens, clone/pull wf-v4.6, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.6"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Data and the loop-2 models (MultiWOZ zip + ids from HF, GLiNER, deberta-v3-base, bge-m3; pinned revisions)
import json, os
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
MODELS = Path("/content/models")
os.environ["CHK_MODELS"] = str(MODELS)          # read by chunking/chk/config.py and s1a/spans.py at import
DATA = Path("/content/attribute-extraction/chunking/data")
for f in ("MULTIWOZ2.4.zip", "conversation_ids.txt"):
    if not (DATA / f).exists():
        hf_hub_download("database-lab/attribute-extraction-data", f"chunking/{f}", repo_type="dataset", local_dir=DATA.parent)
snapshot_download("urchade/gliner_medium-v2.1", revision="40ec419335d09393f298636f471328b722c6da9e", local_dir=MODELS / "gliner_medium-v2.1")
snapshot_download("microsoft/deberta-v3-base", revision="8ccc9b6f36199bec6961081d44eb72fb3f7353f3", local_dir=MODELS / "deberta-v3-base",
                  allow_patterns=["config.json", "spm.model", "tokenizer_config.json"])
snapshot_download("BAAI/bge-m3", revision="5617a9f61b028005a4858fdac845db406aefb181", local_dir=MODELS / "bge-m3")
cfg_path = MODELS / "gliner_medium-v2.1" / "gliner_config.json"   # point GLiNER at the local backbone, as on the laptop
cfg = json.loads(cfg_path.read_text())
cfg["model_name"] = str(MODELS / "deberta-v3-base")
cfg_path.write_text(json.dumps(cfg, indent=2))
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a"); sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
from s1a import spans as SP
g = SP.load()
print("GLiNER ok:", g.inference(["I need a hotel with free wifi."], ["internet access"], threshold=0.3))
del g

In [ ]:
#@title 5. Load the model (gpu_memory_utilization leaves room for bge-m3)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.6  #@param {type:"number"}
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
import time
_t0 = _T_SESSION = time.time()   # the GPU session clock starts with the model load
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)
from s1a import log_v44 as L44
print("model load:", L44.save_load_time("results/wf_v46/model_load.json", time.time() - _t0, MODEL, REVISION))


In [ ]:
#@title 6. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/log_v44.VllmBackend44)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
from s1a import log_v44 as L44
backend = L44.VllmBackend44(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 7. Data check: the Step 1a-2 folders of the raw dev sets (from git)
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import gpu_v46 as G
missing = G.need()
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"


In [ ]:
#@title 8. Encoders and the GLiNER loader (bge-m3 for the Step 3 names; bge-m3 + GLiNER on the GPU for the loop-2 evidence check)
from s1a import step3_v44 as S3, corpus_v45 as CO
from run import encoder
enc3 = S3.bge_m3_encoder()
enc_1a = encoder()
gliner_loader = CO._gliner_on_gpu()[2]
print("encoders ready")


In [ ]:
#@title 9. Y3: loop 1 with LLM entity names (dev50, dev50b) -> results/wf_v46/y3/<set>
MAX_CALLS = 1500  #@param {type:"integer"}
import json, subprocess
res = {}
for dev in G.SETS:
    res[dev] = G.y3(backend, enc3, enc_1a, {dev: G.SETS[dev]}, load_seconds=L44.load_time("results/wf_v46/model_load.json"))[dev]
    print(dev, json.dumps({k: res[dev][k] for k in ('total_llm', 'entity_names', 'step4_chains_finished')}))
    names = json.load(open(G.out_dir(dev) / 'entity_names.json'))
    print('  names:', names['names'], '| groups:', names['groups'])
    assert res[dev]['total_llm']['calls'] <= MAX_CALLS, f'{dev}: more calls than MAX_CALLS: stop and tell Claude'
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v46 Y3 loop 1 with LLM entity names", "wf_v46"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 10. Y5 + Y6: loop 2 on the Y3 output -> results/wf_v46/y3/<set>/loop2
MAX_CALLS = 1500  #@param {type:"integer"}
import json, subprocess
res = G.y56(backend, enc_1a, gliner_loader)
for dev, r in res.items():
    print(dev, (G.out_dir(dev) / 'loop2' / 'report.md').read_text(encoding='utf-8')[:3000])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v46 Y5 Y6 loop 2", "wf_v46"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 11. Y8 (only if time is left): Step 4 v4.3, pass 2 in batches of 20, seed 0 -> results/wf_v46/y3/<set>/step4_p2b20
MAX_CALLS = 1500  #@param {type:"integer"}
import json, subprocess
res = G.y8(backend)
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v46 Y8 step 4 pass 2 batches of 20", "wf_v46"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 12. GPU session time -> results/wf_v46/gpu_session.json, then disconnect
import json, time, subprocess
info = {"gpu_session_seconds": round(time.time() - _T_SESSION, 1), "model_load_seconds": L44.load_time("results/wf_v46/model_load.json")}
open("results/wf_v46/gpu_session.json", "w").write(json.dumps(info, indent=1) + "\n")
print(info)
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v46 GPU session time", "wf_v46"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("Done: Runtime > Disconnect and delete runtime, then tell Claude 'GPU done'")
